# 08 · Normalized Pair Correlation Benchmark

This notebook is a cleaner, more canonical benchmark for pair-correlation behavior in zeta zeros.

It complements the earlier exploratory pair-difference notebooks by building a more standard empirical estimator on an unfolded point sequence and comparing it against:

1. **Poisson** behavior  
2. **finite GUE-matrix** behavior  
3. the standard **GUE reference shape**
   \[
   1 - \left(\frac{\sin \pi s}{\pi s}\right)^2
   \]

## Goal

The goal here is not to produce a theorem-level asymptotic object, but to make the **small-separation repulsion structure** more legible in a cleaner normalized pipeline.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Zeta zeros and local unfolding

In [ ]:
N = 900
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

len(t), len(zeta_gaps)

In [ ]:
def local_normalize_gaps(gaps, window=25):
    kernel = np.ones(window) / window
    local_means = np.convolve(gaps, kernel, mode="same")
    half = window // 2
    for i in range(half):
        local_means[i] = gaps[:i + half + 1].mean()
        local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()
    return gaps / local_means

def unfold_from_gaps(gaps, window=25):
    unfolded_gaps = local_normalize_gaps(gaps, window=window)
    x = np.concatenate([[0.0], np.cumsum(unfolded_gaps)])
    return x, unfolded_gaps

In [ ]:
zeta_x, zeta_unfolded_gaps = unfold_from_gaps(zeta_gaps, window=25)
zeta_unfolded_gaps.mean(), zeta_unfolded_gaps.std()

## Controls: Poisson and finite GUE matrices

In [ ]:
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))
poisson_x, poisson_unfolded_gaps = unfold_from_gaps(poisson_gaps, window=25)

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)
gue_x, gue_unfolded_gaps = unfold_from_gaps(gue_gaps, window=25)

poisson_unfolded_gaps.mean(), gue_unfolded_gaps.mean()

## Nearest-neighbor spacing sanity check

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 34)
plt.hist(zeta_unfolded_gaps, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_unfolded_gaps, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("unfolded spacing")
plt.ylabel("density")
plt.title("Unfolded nearest-neighbor spacing check")
plt.legend()
plt.show()

## Normalized pair-correlation-style estimator

For an unfolded sequence \(x_1,\dots,x_N\), define the positive differences

\[
s = x_j - x_i,\qquad j>i.
\]

We restrict to a finite separation window \(0 < s \le s_{\max}\) and estimate the density of these differences.

To reduce edge bias, we only use base points \(x_i\) that are at least \(s_{\max}\) away from the right boundary of the observed interval.

In [ ]:
def pair_density_estimate(x, s_max=3.0, bins=60):
    x = np.asarray(x)
    L = x[-1] - x[0]

    usable = np.where(x <= x[-1] - s_max)[0]
    diffs = []

    for i in usable:
        d = x[i+1:] - x[i]
        d = d[(d > 0) & (d <= s_max)]
        if len(d):
            diffs.extend(d.tolist())

    diffs = np.array(diffs)
    hist, edges = np.histogram(diffs, bins=bins, range=(0, s_max), density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist, diffs

s_max = 3.0
bins = 60

zeta_s, zeta_hist, zeta_diffs = pair_density_estimate(zeta_x, s_max=s_max, bins=bins)
poisson_s, poisson_hist, poisson_diffs = pair_density_estimate(poisson_x, s_max=s_max, bins=bins)
gue_s, gue_hist, gue_diffs = pair_density_estimate(gue_x, s_max=s_max, bins=bins)

len(zeta_diffs), len(poisson_diffs), len(gue_diffs)

## Pair-density curves

This is the main benchmark plot.

In [ ]:
plt.figure(figsize=(8.8, 5.2))
plt.plot(zeta_s, zeta_hist, linewidth=2, label="zeta")
plt.plot(poisson_s, poisson_hist, linewidth=2, label="Poisson")
plt.plot(gue_s, gue_hist, linewidth=2, label="GUE matrix")
plt.xlabel("pair separation")
plt.ylabel("estimated density")
plt.title("Normalized pair-density benchmark")
plt.legend()
plt.show()

## Small-separation focus

This is the most important region:
- Poisson should remain comparatively higher near 0
- GUE-like behavior should be suppressed near 0
- zeta is expected to track GUE more closely than Poisson

In [ ]:
mask = zeta_s <= 1.2

plt.figure(figsize=(8.8, 5.0))
plt.plot(zeta_s[mask], zeta_hist[mask], linewidth=2, label="zeta")
plt.plot(poisson_s[mask], poisson_hist[mask], linewidth=2, label="Poisson")
plt.plot(gue_s[mask], gue_hist[mask], linewidth=2, label="GUE matrix")
plt.xlabel("pair separation")
plt.ylabel("estimated density")
plt.title("Small-separation normalized pair density")
plt.legend()
plt.show()

## Overlay with the standard GUE reference shape

The standard GUE pair-correlation reference is

\[
1 - \left(\frac{\sin \pi s}{\pi s}\right)^2.
\]

To compare on the same finite interval, we normalize that curve on \([0,s_{\max}]\).

In [ ]:
s = np.linspace(1e-3, s_max, 1000)
gue_reference = 1 - (np.sin(np.pi * s) / (np.pi * s))**2
gue_reference_norm = gue_reference / np.trapz(gue_reference, s)

plt.figure(figsize=(8.8, 5.2))
plt.plot(zeta_s, zeta_hist, linewidth=2, label="zeta")
plt.plot(gue_s, gue_hist, linewidth=2, label="GUE matrix")
plt.plot(s, gue_reference_norm, linewidth=2, label="normalized GUE reference")
plt.xlabel("pair separation")
plt.ylabel("density / normalized shape")
plt.title("Pair-density benchmark with GUE reference")
plt.legend()
plt.show()

## Very-small-separation zoom

This makes the near-zero repulsion region easier to inspect.

In [ ]:
mask_tiny = zeta_s <= 0.6

plt.figure(figsize=(8.8, 5.0))
plt.plot(zeta_s[mask_tiny], zeta_hist[mask_tiny], linewidth=2, label="zeta")
plt.plot(poisson_s[mask_tiny], poisson_hist[mask_tiny], linewidth=2, label="Poisson")
plt.plot(gue_s[mask_tiny], gue_hist[mask_tiny], linewidth=2, label="GUE matrix")
plt.xlabel("pair separation")
plt.ylabel("estimated density")
plt.title("Very-small-separation normalized pair density")
plt.legend()
plt.show()

## Area-under-curve check on the finite interval

In [ ]:
bin_width = zeta_s[1] - zeta_s[0]
areas = {
    "zeta": float(np.sum(zeta_hist) * bin_width),
    "Poisson": float(np.sum(poisson_hist) * bin_width),
    "GUE": float(np.sum(gue_hist) * bin_width),
}
areas

## Quantitative summaries

We summarize:
- mean unfolded spacing
- pair-density value near the first few bins
- total sample counts in the pair-difference pool

In [ ]:
summary = {
    "unfolded_gap_mean": {
        "zeta": float(zeta_unfolded_gaps.mean()),
        "Poisson": float(poisson_unfolded_gaps.mean()),
        "GUE": float(gue_unfolded_gaps.mean()),
    },
    "pair_density_first_bin": {
        "zeta": float(zeta_hist[0]),
        "Poisson": float(poisson_hist[0]),
        "GUE": float(gue_hist[0]),
    },
    "pair_density_second_bin": {
        "zeta": float(zeta_hist[1]),
        "Poisson": float(poisson_hist[1]),
        "GUE": float(gue_hist[1]),
    },
    "pair_samples": {
        "zeta": int(len(zeta_diffs)),
        "Poisson": int(len(poisson_diffs)),
        "GUE": int(len(gue_diffs)),
    },
}
summary

## Notes

- This notebook is still finite-sample and practical; it is not the full asymptotic Montgomery pair-correlation object.
- The normalization is cleaner than the earlier pair-difference histograms because it uses an unfolded coordinate and a fixed finite separation window with a simple edge restriction.
- The most important benchmark is whether zeta tracks the finite GUE matrix reference more closely than the Poisson control near small separation.
- The normalized analytic GUE curve is included as a reference shape, not as an exact finite-sample fit.

## Next directions

A natural next notebook could do one of these:

1. compare several unfolding windows for robustness  
2. increase matrix size and sample count for the GUE benchmark  
3. condition the higher-order 4-gap / 5-gap windows on unusually small or large local gaps  
4. combine the canonical pair-correlation benchmark with the higher-order local descriptors